# 额外的周末练习 - 第 2 周

现在，使用您从第 2 周学到的所有知识为您在第 1 周练习中构建的技术问题/回答器构建完整的原型。

这应该包括 Gradio UI、流媒体、使用系统提示来添加专业知识以及在模型之间切换的能力。如果您能够演示工具的使用，则可获得奖励积分！

如果您觉得大胆，请看看是否可以添加音频输入，以便您可以与它交谈，并让它用音频进行响应。 ChatGPT 或 Claude 可以帮助您，如果您有疑问，也可以给我发电子邮件。

我很快就会在这里发布完整的解决方案 - 除非有人比我先一步......

这方面的商业应用有很多，从语言导师到公司入职解决方案，再到人工智能伴侣和课程（就像这个！），我迫不及待地想看到你的结果。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
import os
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from datetime import datetime

In [ ]:
load_dotenv()
# 如果不使用 .env，请在此处手动设置：
# os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-your-key-here"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def init_db():
    conn = sqlite3.connect("research_agent.db")
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS logs 
                      (timestamp TEXT, model TEXT, prompt TEXT, response TEXT)''')
    conn.commit()
    conn.close()

def log_interaction(model, prompt, response):
    conn = sqlite3.connect("research_agent.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO logs VALUES (?, ?, ?, ?)", 
                   (datetime.now().strftime("%Y-%m-%d %H:%M:%S"), model, prompt, response))
    conn.commit()
    conn.close()

init_db()
print("✅ Database and Client Initialized.")

In [ ]:
import json

# 模拟技术数据库搜索
def search_tech_docs(query):
    """Searches the internal technical manual for documentation."""
    docs = {
        "api": "API stands for Application Programming Interface. It allows software to talk.",
        "gradio": "Gradio is an open-source Python library used to build ML web apps.",
        "sqlite": "SQLite is a C-language library that implements a small, fast SQL database engine."
    }
    # 如果找到关键字则返回文档，否则返回默认消息
    for key in docs:
        if key in query.lower():
            return docs[key]
    return "No specific documentation found for this topic."

# 定义 OpenAI/OpenRouter 格式的工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tech_docs",
            "description": "Search the internal technical manual for coding definitions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The technical topic to look up."}
                },
                "required": ["query"],
            },
        },
    }
]

In [ ]:
import json

# 模拟技术数据库搜索
def search_tech_docs(query):
    """Searches the internal technical manual for documentation."""
    docs = {
        "api": "API stands for Application Programming Interface. It allows software to talk.",
        "gradio": "Gradio is an open-source Python library used to build ML web apps.",
        "sqlite": "SQLite is a C-language library that implements a small, fast SQL database engine."
    }
    # 如果找到关键字则返回文档，否则返回默认消息
    for key in docs:
        if key in query.lower():
            return docs[key]
    return "No specific documentation found for this topic."

# 定义 OpenAI/OpenRouter 格式的工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tech_docs",
            "description": "Search the internal technical manual for coding definitions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The technical topic to look up."}
                },
                "required": ["query"],
            },
        },
    }
]

In [ ]:
def agent_chat(message, history, model_name, system_prompt):
    models = {
        "Claude 3.5 Sonnet": "anthropic/claude-3.5-sonnet",
        "GPT-4o": "openai/gpt-4o",
        "Gemini 1.5 Pro": "google/gemini-pro-1.5"
    }
    
    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})

    # 第 1 步：首次调用以查看是否需要工具
    response = client.chat.completions.create(
        model=models[model_name],
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # 步骤 2：处理工具调用（如果存在）
    if tool_calls:
        for tool_call in tool_calls:
            function_args = json.loads(tool_call.function.arguments)
            observation = search_tech_docs(function_args.get("query"))
            
            messages.append(response_message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "search_tech_docs",
                "content": observation,
            })
    
    # 第 3 步：传输最终响应
    final_stream = client.chat.completions.create(
        model=models[model_name],
        messages=messages,
        stream=True
    )

    full_text = ""
    for chunk in final_stream:
        if chunk.choices[0].delta.content:
            full_text += chunk.choices[0].delta.content
            yield full_text
            
    # 第 4 步：登录 SQLite
    log_interaction(model_name, message, full_text)

In [ ]:
import pandas as pd
import sqlite3
import gradio as gr

# 从 SQLite 为 Gradio UI 提取数据的函数
def get_history():
    try:
        conn = sqlite3.connect("research_agent.db")
        df = pd.read_sql_query("SELECT * FROM logs ORDER BY timestamp DESC", conn)
        conn.close()
        return df
    except Exception as e:
        return pd.DataFrame({"Status": ["No history found yet. Start a chat!"]})

# --- Gradio UI 布局（第 5 天：块和选项卡）---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("#  Research & Logging Agent")
    gr.Markdown("### Week 2 Exercise")
    
    with gr.Tabs():
        # TAB 1：人工智能研究员
        with gr.TabItem("AI Researcher"):
            with gr.Row():
                model_sel = gr.Dropdown(
                    choices=["Claude 3.5 Sonnet", "GPT-4o", "Gemini 1.5 Pro"], 
                    value="Claude 3.5 Sonnet", 
                    label="Agent Brain"
                )
                sys_prompt = gr.Textbox(
                    value="You are a Senior Technical Researcher. Use your tools to provide accurate data.", 
                    label="System Instructions"
                )
            
            # 流媒体聊天界面
            gr.ChatInterface(
                fn=agent_chat, 
                additional_inputs=[model_sel, sys_prompt], 
                type="tuples"
            )
            
        # TAB 2：数据库日志（固定缩进）
        with gr.TabItem("Database Logs"):
            gr.Markdown("### 📜 Conversation History")
            gr.Markdown("Data below is pulled in real-time from `research_agent.db`.")
            
            refresh_btn = gr.Button("🔄 Refresh History", variant="primary")
            
            # 将 SQLite 数据显示为干净的表
            history_table = gr.DataFrame(value=get_history)
            
            # 将按钮链接到刷新功能
            refresh_btn.click(fn=get_history, outputs=history_table)

# 启动应用程序
demo.launch(share=True, debug=True)